In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("/home/patel-krish/anaconda3/BrainyBeam/heart.csv")

In [3]:
X = df.drop(columns=["target"]).to_numpy(dtype=float)
y = df["target"].to_numpy()
features = df.drop(columns=["target"]).columns.tolist()

In [4]:
def gini(labels):
    if len(labels) == 0:
        return 0

    _, counts = np.unique(labels, return_counts=True)
    probabilities = counts / len(labels)

    return 1 - np.sum(probabilities ** 2)


importance = {feature: 0.0 for feature in features}

In [5]:
def best_split(indices):
    labels = y[indices]
    parent_gini = gini(labels)
    best = None
    n = len(indices)

    for j, feature in enumerate(features):
        values = X[indices, j]
        unique_values = np.unique(values)

        if len(unique_values) < 2:
            continue

        thresholds = (
            unique_values[:-1] + unique_values[1:]
        ) / 2

        for threshold in thresholds:
            mask = values <= threshold
            left_count = mask.sum()
            right_count = n - left_count

            if left_count < 10 or right_count < 10:
                continue

            left_y = labels[mask]
            right_y = labels[~mask]

            weighted_gini = (
                (left_count / n) * gini(left_y)
                + (right_count / n) * gini(right_y)
            )

            gain = parent_gini - weighted_gini

            if best is None or gain > best[0]:
                best = (
                    gain, j, threshold,
                    indices[mask], indices[~mask]
                )

    return best

In [6]:
def build_tree(indices, depth=0, max_depth=4):
    labels = y[indices]

    if (
        depth >= max_depth
        or len(indices) < 20
        or len(np.unique(labels)) < 2
    ):
        return

    split = best_split(indices)

    if split is None or split[0] <= 0:
        return

    gain, feature_index, threshold, left, right = split

    feature = features[feature_index]

    importance[feature] += (
        len(indices) / len(y)
    ) * gain

    print(
        f"Depth: {depth}, Feature: {feature}, "
        f"Threshold: {threshold:.2f}, "
        f"Gini reduction: {gain:.4f}"
    )

    build_tree(left, depth + 1, max_depth)
    build_tree(right, depth + 1, max_depth)

In [7]:
build_tree(np.arange(len(y)))

total_importance = sum(importance.values())

if total_importance > 0:
    for feature in importance:
        importance[feature] /= total_importance

Depth: 0, Feature: cp, Threshold: 0.50, Gini reduction: 0.1349
Depth: 1, Feature: ca, Threshold: 0.50, Gini reduction: 0.0931
Depth: 2, Feature: thal, Threshold: 2.50, Gini reduction: 0.1628
Depth: 3, Feature: exang, Threshold: 0.50, Gini reduction: 0.0910
Depth: 3, Feature: oldpeak, Threshold: 0.65, Gini reduction: 0.1035
Depth: 2, Feature: trestbps, Threshold: 109.00, Gini reduction: 0.0116
Depth: 3, Feature: chol, Threshold: 301.50, Gini reduction: 0.0044
Depth: 1, Feature: age, Threshold: 56.50, Gini reduction: 0.0461
Depth: 2, Feature: oldpeak, Threshold: 1.95, Gini reduction: 0.0255
Depth: 3, Feature: thalach, Threshold: 142.50, Gini reduction: 0.0154
Depth: 3, Feature: age, Threshold: 43.50, Gini reduction: 0.0450
Depth: 2, Feature: sex, Threshold: 0.50, Gini reduction: 0.0923
Depth: 3, Feature: age, Threshold: 62.50, Gini reduction: 0.0756
Depth: 3, Feature: chol, Threshold: 245.50, Gini reduction: 0.1003


In [8]:
ranking = pd.DataFrame({
    "Feature": list(importance.keys()),
    "Importance": list(importance.values())
})

ranking = ranking.sort_values(
    by="Importance", ascending=False
).reset_index(drop=True)

ranking["Importance (%)"] = (
    ranking["Importance"] * 100
)

print("\nFeature Importance Ranking")
print(ranking.round(4).to_string(index=False))



Feature Importance Ranking
 Feature  Importance  Importance (%)
      cp      0.4296         42.9613
      ca      0.1438         14.3783
    thal      0.1108         11.0770
     age      0.0964          9.6432
     sex      0.0597          5.9652
 oldpeak      0.0556          5.5598
    chol      0.0444          4.4370
   exang      0.0353          3.5325
 thalach      0.0144          1.4399
trestbps      0.0101          1.0058
 restecg      0.0000          0.0000
     fbs      0.0000          0.0000
   slope      0.0000          0.0000
